In [2]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df_train = pd.read_csv('../datasets/clean.csv')
df_test = pd.read_csv('../datasets/power_leak.csv')

eps_features = ['net_power', 'total_drain', 'solar_input', 'is_scrubber_on', 'is_engine_on']

X_train = df_train[eps_features]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
model.fit(X_train_scaled)

df_test_filtered = df_test[df_test['timestamp'] > 100].copy()
X_test_scaled = scaler.transform(df_test_filtered[eps_features])

df_test_filtered['scores'] = model.decision_function(X_test_scaled)
df_test_filtered['anomaly_pred'] = model.predict(X_test_scaled)


In [10]:
print(df_test_filtered[['timestamp', 'net_power', 'total_drain', 'solar_input', 'is_scrubber_on', 'is_engine_on']][df_test_filtered['is_anomaly'] == 1].iloc[0])

timestamp           671.000000
net_power          7899.892076
total_drain        3300.000000
solar_input       11199.892076
is_scrubber_on        0.000000
is_engine_on          1.000000
Name: 662, dtype: float64


In [27]:
print(len(df_test_filtered[['timestamp', 'solar_input', 'is_scrubber_on', 'is_engine_on']][df_test_filtered['anomaly_pred'] == -1]))
print(df_test_filtered[['timestamp', 'solar_input', 'scores']][df_test_filtered['scores'] < 0])

154
      timestamp   solar_input    scores
243         252  11199.933507 -0.018926
599         608  11199.932213 -0.021922
600         609  11199.922637 -0.030476
601         610  11199.926149 -0.030476
603         612  11199.927288 -0.030476
...         ...           ...       ...
994        1003  11199.936770 -0.026010
995        1004  11199.935198 -0.027520
996        1005  11199.946568 -0.000792
1020       1029  11199.932744 -0.025432
1021       1030  11199.919633 -0.030476

[154 rows x 3 columns]
